# 04 — Filter & fusion decision (combined)

Single self-contained notebook merging the former 06 (batch-effect analysis + filter exploration) and the three probes 06b/06c/06d (M1 clinical, M3 wavelet, M7 deep) plus 06e (synthesis). Nothing removed; rigor/cleanup added (English, guards, abandoned approach kept & labeled).

**Decision frozen:** bandpass Butterworth order 4, 0.5-40 Hz, zero-phase; PTB-XL+Ningbo fusion validated.

**Inputs:** raw signals via `signal_loading`, `metadata_combined.csv`.
**Outputs:** `data/processed/filter_config.json`, probe CSVs in `data/processed/filter_probes/`, figures.

> Heavy probe extractions/training are GUARDED (skip if cached CSVs exist). Legacy 06/06b/c/d/e kept as reference; this notebook supersedes them (final renumbering later). Narrative: see `JOURNAL.md`.

### Block 4.0: Setup

In [ ]:
# Filter & fusion decision. Single source of truth for the batch-effect analysis, the bandpass
# filter choice (frozen 0.5-40 Hz) and the validation that PTB-XL + Ningbo can be fused.
# Heavy probe extractions/training are guarded (skip if their cached CSVs exist).
from pathlib import Path
import os, sys, json
import numpy as np, pandas as pd

ROOT        = Path(r".")
PTBXL_BASE  = ROOT / "data/raw/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
NINGBO_ROOT = ROOT / "data/raw/ningbo/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0"
PROCESSED   = ROOT / "data/processed"
PROBES      = PROCESSED / "filter_probes"
FIG         = ROOT / "reports/figures"
PROBES.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))

FS = 500
FORCE_REEXTRACT = False   # re-run guard for the heavy probe extractions / CNN training

meta = pd.read_csv(PROCESSED / "metadata_combined.csv", dtype={'ecg_id': str})
print("=== Setup ===")
print(f"  metadata_combined: {len(meta)} ECGs | WPW {int((meta.label==1).sum())}")
print(f"  by source: {meta.groupby('source').size().to_dict()}")

### Block 4.1: Cross-dataset loading sanity check

In [ ]:
# Cross-dataset loading sanity check: both datasets must load to (5000, 12), 500 Hz, mV, 0 NaN,
# with the same lead order. (Path-resolution debugging iterations from the original 06 are folded
# into this single working check.)
import wfdb
LEADS_STD = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
ningbo = pd.read_csv(PROCESSED / "metadata_ningbo.csv", dtype={'ecg_id': str})
ningbo_path = dict(zip(ningbo['ecg_id'], ningbo['rel_path']))

def resolve_path(ecg_id, source):
    if 'ptb' in str(source).lower():
        k = int(ecg_id); folder = f"{(k//1000)*1000:05d}"
        return str(PTBXL_BASE / "records500" / folder / f"{k:05d}_hr")
    p = NINGBO_ROOT / ningbo_path[str(ecg_id)]
    return str(p.with_suffix('')) if p.suffix else str(p)

rows = []
for s in meta['source'].unique():
    for eid in meta[meta['source'] == s]['ecg_id'].head(5):
        rec = wfdb.rdrecord(resolve_path(eid, s)); sig = rec.p_signal
        rows.append({'source': s, 'shape': sig.shape, 'fs': rec.fs,
                     'leads_std': [n.upper() for n in rec.sig_name] == [l.upper() for l in LEADS_STD],
                     'n_nan': int(np.isnan(sig).sum()), 'amp_med': round(float(np.nanmedian(np.abs(sig))), 4)})
insp = pd.DataFrame(rows)
print(insp.to_string(index=False))
assert all(insp['shape'] == (5000, 12)) and all(insp['fs'] == 500) and insp['n_nan'].sum() == 0
print("[OK] uniform shape/fs/units, same lead order, 0 NaN across datasets.")

### Block 4.2: Canonical loader (import + verify, never regenerate)

In [ ]:
# Canonical loader: import the committed module src/signal_loading.py (single source of truth used
# by every model and by Flask). NOTE: the module is a versioned source file shipped with the repo
# (like requirements.txt) -- it is imported here, never regenerated from the notebook.
import importlib, signal_loading
importlib.reload(signal_loading)
from signal_loading import load_signal, LEADS_CANONICAL
print("Imported signal_loading | canonical lead order:", LEADS_CANONICAL)
for s in meta['source'].unique():
    eid = meta[meta['source'] == s]['ecg_id'].iloc[0]
    sig = load_signal(eid, s)
    print(f"  {s}/{eid}: shape {sig.shape}, dtype {sig.dtype}, NaN {int(np.isnan(sig).sum())}")

### Block 4.3: Batch effect — spectral band powers

In [ ]:
# Batch effect, step 1: relative spectral band powers (Welch) on a stratified sample
# (1500 non-WPW/dataset + all WPW), leads II/V1/V5. Guarded by the cached CSV.
from scipy.signal import welch
from tqdm import tqdm
SPEC_CSV = PROCESSED / "filter_batcheffect_spectra.csv"
LEADS_MEASURE = ['II', 'V1', 'V5']
LEAD_IDX = {'I':0,'II':1,'III':2,'AVR':3,'AVL':4,'AVF':5,'V1':6,'V2':7,'V3':8,'V4':9,'V5':10,'V6':11}
BANDS = {'baseline_0.5':(0.0,0.5),'low_0.5_15':(0.5,15),'delta_wpw_15_40':(15,40),
         'mains_50':(48,52),'hf_40_100':(40,100),'noise_100_250':(100,250)}

if SPEC_CSV.exists() and not FORCE_REEXTRACT:
    spec = pd.read_csv(SPEC_CSV, dtype={'ecg_id': str})
    print(f"{SPEC_CSV.name} exists -> SKIPPED spectral extraction (loaded {len(spec)} rows).")
else:
    rng = np.random.RandomState(42); parts = []
    for s in meta['source'].unique():
        sub = meta[meta['source'] == s]
        parts.append(sub[sub.label == 0].sample(n=min(1500, int((sub.label==0).sum())), random_state=42))
        parts.append(sub[sub.label == 1])
    sample = pd.concat(parts).reset_index(drop=True)
    def band_powers(x):
        f, pxx = welch(x, fs=FS, nperseg=2048, noverlap=1024); tot = np.trapezoid(pxx, f)
        if tot <= 0: return {b: np.nan for b in BANDS}
        return {n: np.trapezoid(pxx[(f>=lo)&(f<hi)], f[(f>=lo)&(f<hi)])/tot for n,(lo,hi) in BANDS.items()}
    rows = []
    for _, r in tqdm(sample.iterrows(), total=len(sample), desc="Spectra", unit="ecg"):
        try: sig = load_signal(r['ecg_id'], r['source'])
        except Exception: continue
        for lead in LEADS_MEASURE:
            bp = band_powers(sig[:, LEAD_IDX[lead]])
            bp.update({'ecg_id': r['ecg_id'], 'source': r['source'], 'label': r['label'], 'lead': lead})
            rows.append(bp)
    spec = pd.DataFrame(rows); spec.to_csv(SPEC_CSV, index=False)
    print(f"Computed spectra: {len(spec)} rows -> {SPEC_CSV.name}")
print("\nMean relative band power (non-WPW), by source:")
print(spec[spec.label==0].groupby('source')[list(BANDS)].mean().round(4).to_string())

### Block 4.4: Batch effect — Cohen's d & dataset-separation AUC

In [ ]:
# Batch effect, step 2: Cohen's d (ningbo vs ptbxl) and dataset-separation AUC per band/lead.
# Measures HOW separable the two datasets are from spectral shape alone (the batch effect).
from sklearn.metrics import roc_auc_score
spec = pd.read_csv(PROCESSED / "filter_batcheffect_spectra.csv", dtype={'ecg_id': str})
BANDS = ['baseline_0.5','low_0.5_15','delta_wpw_15_40','mains_50','hf_40_100','noise_100_250']

def cohens_d(a, b):
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]; na, nb = len(a), len(b)
    if na < 2 or nb < 2: return np.nan
    sp = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na+nb-2))
    return (a.mean() - b.mean()) / sp if sp > 0 else np.nan
def sep_auc(vals, is_nin):
    m = ~np.isnan(vals)
    if m.sum() < 10 or len(np.unique(is_nin[m])) < 2: return np.nan
    a = roc_auc_score(is_nin[m], vals[m]); return max(a, 1 - a)

def analyse(df, title):
    print(f"\n{title}")
    rows = []
    for lead in sorted(df['lead'].unique()):
        sub = df[df['lead'] == lead]; is_nin = (sub['source'] == 'ningbo').values.astype(int)
        nin, ptb = sub[sub.source=='ningbo'], sub[sub.source=='ptbxl']
        for band in BANDS:
            rows.append({'lead':lead,'band':band,
                         'cohens_d':cohens_d(nin[band].values, ptb[band].values),
                         'sep_AUC':sep_auc(sub[band].values, is_nin)})
    res = pd.DataFrame(rows)
    synth = res.assign(absd=res.cohens_d.abs()).groupby('band')['absd'].mean().reindex(BANDS)
    print("  mean |Cohen's d| per band:", {b: round(float(v),2) for b,v in synth.items()})
    print("  max dataset-separation AUC:", round(float(res.sep_AUC.max()),3))
    return res

res_neg = analyse(spec[spec.label==0], "BATCH EFFECT — non-WPW (reference)")
_ = analyse(spec[spec.label==1], "CHECK — WPW only")
print("\nRead: no single band fully separates the datasets, but separation is strong overall "
      "(V1 high-freq ~0.8). The batch effect is real -> the true test is cross-dataset generalization.")

### Block 4.5a: Harmonization attempt (ABANDONED — circular, kept for transparency)

In [ ]:
# Harmonization attempt — ABANDONED (kept for transparency). Measuring per-band dataset separation
# AFTER filtering is CIRCULAR: relative-power renormalization on tiny residuals creates artifacts
# (e.g. the baseline band inflates to ~0.68 with 0.5-40 due to edge distortion). Superseded by 4.5b.
from scipy.signal import welch, butter, sosfiltfilt
from sklearn.metrics import roc_auc_score
LEAD_IDX = {'I':0,'II':1,'III':2,'AVR':3,'AVL':4,'AVF':5,'V1':6,'V2':7,'V3':8,'V4':9,'V5':10,'V6':11}
BANDS = {'baseline_0.5':(0.0,0.5),'low_0.5_15':(0.5,15),'delta_wpw_15_40':(15,40),
         'mains_50':(48,52),'hf_40_100':(40,100),'noise_100_250':(100,250)}
FILTERS = {'no_filter':None,'bp_0.5_40':(0.5,40),'bp_1_40':(1.0,40),'bp_0.5_100':(0.5,100)}
SOS = {k:(butter(4,[v[0]/(FS/2),v[1]/(FS/2)],btype='band',output='sos') if v else None) for k,v in FILTERS.items()}
print("Block 4.5a is intentionally kept as documentation of an abandoned (circular) approach.")
print("The trustworthy harmonization measurement is Block 4.5b (dataset classifier on absolute features).")

### Block 4.5b: Harmonization test (corrected — dataset classifier)

In [ ]:
# Harmonization test (corrected): for each candidate filter, train a 'ningbo vs ptbxl' classifier on
# ABSOLUTE log-spectral + temporal features (non-circular) and measure CV separation AUC.
# RESULT: the batch effect is ~0.95 and NO bandpass harmonizes it (drop ~0.05 only) -> filtering is
# denoising, not harmonization. The real judge of fusion becomes cross-dataset generalization (probes).
from scipy.signal import welch, butter, sosfiltfilt
from scipy.integrate import trapezoid
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from tqdm import tqdm
LEAD_IDX = {'I':0,'II':1,'III':2,'AVR':3,'AVL':4,'AVF':5,'V1':6,'V2':7,'V3':8,'V4':9,'V5':10,'V6':11}
LEADS = ['II','V1','V5']
FEAT_BANDS = [(0.5,4),(4,8),(8,15),(15,25),(25,40),(40,60),(60,100),(100,250)]
FILTERS = {'no_filter':None,'bp_0.5_40':(0.5,40),'bp_1_40':(1.0,40),'bp_0.5_100':(0.5,100)}
SOS = {k:(butter(4,[v[0]/(FS/2),v[1]/(FS/2)],btype='band',output='sos') if v else None) for k,v in FILTERS.items()}

def feats(x):
    f, pxx = welch(x, fs=FS, nperseg=2048, noverlap=1024)
    out = [np.log10(trapezoid(pxx[(f>=lo)&(f<hi)], f[(f>=lo)&(f<hi)])+1e-12) for lo,hi in FEAT_BANDS]
    return out + [x.std(), np.sqrt(np.mean(np.diff(x)**2)), np.mean(np.abs(x))]

neg = {s: meta[(meta.source==s)&(meta.label==0)] for s in meta['source'].unique()}
n_per = min(1500, *[len(v) for v in neg.values()])
sample = pd.concat([v.sample(n=n_per, random_state=42) for v in neg.values()]).reset_index(drop=True)
print(f"Balanced non-WPW sample: {n_per}/dataset. Loading signals once...")
sigs, yds = [], []
for _, r in tqdm(sample.iterrows(), total=len(sample), unit="ecg"):
    try: sigs.append(load_signal(r['ecg_id'], r['source']).astype(np.float64)); yds.append(1 if r['source']=='ningbo' else 0)
    except Exception: pass
yds = np.array(yds)

results = []
for fname, sos in SOS.items():
    X = []
    for sig in sigs:
        row = []
        for lead in LEADS:
            s = np.ascontiguousarray(sig[:, LEAD_IDX[lead]], dtype=np.float64)
            s = s if sos is None else sosfiltfilt(sos, s)
            row += feats(s)
        X.append(row)
    clf = HistGradientBoostingClassifier(max_depth=3, max_iter=200, learning_rate=0.05, random_state=42)
    cv = StratifiedKFold(5, shuffle=True, random_state=42)
    auc = cross_val_score(clf, np.array(X), yds, cv=cv, scoring='roc_auc').mean()
    results.append({'filter': fname, 'sep_AUC': auc})
res = pd.DataFrame(results).sort_values('sep_AUC')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(res.to_string(index=False))
base = res[res['filter']=='no_filter'].sep_AUC.iloc[0]
print(f"\nNo filter: {base:.4f} (starting batch effect). Best filter still ~{res.sep_AUC.min():.2f} "
      f"-> NO bandpass harmonizes the batch effect (denoising only).")

### Block 4.6: Batch effect vs the WPW label (fusion-viability test)

In [ ]:
# Is the batch effect aligned with the WPW label? (decisive fusion-viability test)
# Train 'ningbo vs ptbxl' on WPW-only vs on a size-matched non-WPW set, under the 0.5-40 filter.
# RESULT: WPW are LESS dataset-separable than non-WPW (~0.83 < ~0.89) -> the pathological signal
# partly dominates the batch effect (good sign). Still high in absolute terms -> cross-dataset must be proven.
from scipy.signal import welch, butter, sosfiltfilt
from scipy.integrate import trapezoid
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from tqdm import tqdm
LEAD_IDX = {'I':0,'II':1,'III':2,'AVR':3,'AVL':4,'AVF':5,'V1':6,'V2':7,'V3':8,'V4':9,'V5':10,'V6':11}
LEADS = ['II','V1','V5']; FEAT_BANDS = [(0.5,4),(4,8),(8,15),(15,25),(25,40),(40,60),(60,100),(100,250)]
SOS = butter(4, [0.5/(FS/2), 40/(FS/2)], btype='band', output='sos')

def feats_signal(ecg_id, source):
    sig = load_signal(ecg_id, source).astype(np.float64); row = []
    for lead in LEADS:
        s = sosfiltfilt(SOS, np.ascontiguousarray(sig[:, LEAD_IDX[lead]], dtype=np.float64))
        f, pxx = welch(s, fs=FS, nperseg=2048, noverlap=1024)
        row += [np.log10(trapezoid(pxx[(f>=lo)&(f<hi)], f[(f>=lo)&(f<hi)])+1e-12) for lo,hi in FEAT_BANDS]
        row += [s.std(), np.sqrt(np.mean(np.diff(s)**2)), np.mean(np.abs(s))]
    return row

def build_X(df):
    X, yd = [], []
    for _, r in df.iterrows():
        try: X.append(feats_signal(r['ecg_id'], r['source'])); yd.append(1 if r['source']=='ningbo' else 0)
        except Exception: pass
    return np.array(X), np.array(yd)
def sep_auc_cv(X, yd, n_rep=10):
    a = []
    for rep in range(n_rep):
        cv = StratifiedKFold(5, shuffle=True, random_state=rep)
        clf = HistGradientBoostingClassifier(max_depth=3, max_iter=150, learning_rate=0.05, random_state=rep)
        a.extend(cross_val_score(clf, X, yd, cv=cv, scoring='roc_auc'))
    return np.array(a)

wpw = meta[meta.label==1]
print("Features — WPW (142)..."); Xw, yw = build_X(wpw); auc_w = sep_auc_cv(Xw, yw)
neg = meta[meta.label==0]
neg_bal = pd.concat([neg[neg.source=='ningbo'].sample(n=int((yw==1).sum()), random_state=0),
                     neg[neg.source=='ptbxl'].sample(n=int((yw==0).sum()), random_state=0)])
print("Features — matched non-WPW..."); Xn, yn = build_X(neg_bal); auc_n = sep_auc_cv(Xn, yn)
rng = np.random.RandomState(42); null = []
for _ in tqdm(range(200), desc="Permutation", unit="perm"):
    yp = rng.permutation(yw)
    null.append(cross_val_score(HistGradientBoostingClassifier(max_depth=3,max_iter=150,learning_rate=0.05,random_state=0),
                Xw, yp, cv=StratifiedKFold(5,shuffle=True,random_state=0), scoring='roc_auc').mean())
p_val = (np.array(null) >= auc_w.mean()).mean()
print(f"\nDataset separation on WPW only : AUC {auc_w.mean():.4f} +/- {auc_w.std():.4f}")
print(f"Dataset separation on non-WPW  : AUC {auc_n.mean():.4f} +/- {auc_n.std():.4f}")
print(f"Permutation test (WPW): null median {np.median(null):.3f}, p = {p_val:.3f}")
print("Read: WPW less dataset-separable than non-WPW -> WPW signal partly dominates the batch effect.")

### Block 4.7: Delta-wave preservation under the filter

In [ ]:
# Delta-wave preservation: does the 0.5-40 filter keep the WPW delta wave? 3 WPW/dataset, leads V1/V2,
# raw vs 0.5-40 (ref 0.5-100). Quantifies R-amplitude loss and QRS-upslope loss (the delta carrier).
from scipy.signal import butter, sosfiltfilt
import matplotlib.pyplot as plt
LEAD_IDX = {'I':0,'II':1,'III':2,'AVR':3,'AVL':4,'AVF':5,'V1':6,'V2':7,'V3':8,'V4':9,'V5':10,'V6':11}
sos_40  = butter(4, [0.5/(FS/2), 40/(FS/2)],  btype='band', output='sos')
sos_100 = butter(4, [0.5/(FS/2), 100/(FS/2)], btype='band', output='sos')
filt = lambda s, sos: sosfiltfilt(sos, np.ascontiguousarray(s, dtype=np.float64))
wpw = meta[meta.label==1]
picks = []
for s in ['ptbxl','ningbo']:
    picks += wpw[wpw.source==s].head(3)[['ecg_id','source']].values.tolist()

LEADS_SHOW = ['V1','V2']
fig, axes = plt.subplots(len(picks), len(LEADS_SHOW), figsize=(15, 2.6*len(picks)))
for i,(eid,src) in enumerate(picks):
    sig = load_signal(eid, src).astype(np.float64)
    for j,lead in enumerate(LEADS_SHOW):
        raw = sig[:, LEAD_IDX[lead]]; f40, f100 = filt(raw, sos_40), filt(raw, sos_100)
        rpk = int(np.argmax(np.abs(f40[500:2500]))) + 500; a, b = max(0,rpk-200), min(len(raw),rpk+600)
        t = np.arange(a,b)/FS; ax = axes[i,j]
        ax.plot(t, raw[a:b], color='0.75', lw=0.8, label='raw')
        ax.plot(t, f100[a:b], color='#16a34a', lw=1.0, alpha=0.8, label='0.5-100')
        ax.plot(t, f40[a:b], color='#2563eb', lw=1.4, label='0.5-40')
        ax.set_title(f"{src}/{eid} — {lead}", fontsize=9); ax.grid(alpha=.3)
        if i==0 and j==0: ax.legend(fontsize=7)
fig.suptitle("Delta wave: raw vs 0.5-40 Hz (ref 0.5-100) — known WPW", y=1.005); fig.tight_layout()
fig.savefig(FIG / "filter_delta_inspection.png", dpi=150, bbox_inches='tight'); plt.show()

rows = []
for eid,src in picks:
    sig = load_signal(eid, src).astype(np.float64)
    for lead in LEADS_SHOW:
        raw = sig[:, LEAD_IDX[lead]]; f40, f100 = filt(raw, sos_40), filt(raw, sos_100)
        rpk = int(np.argmax(np.abs(f100[500:2500])))+500
        sl40 = np.max(np.abs(np.diff(f40[rpk-20:rpk])))*FS; sl100 = np.max(np.abs(np.diff(f100[rpk-20:rpk])))*FS
        rows.append({'amp_loss_%': round((f100[rpk]-f40[rpk])/abs(f100[rpk])*100,1),
                     'slope_loss_%': round((sl100-sl40)/abs(sl100)*100,1)})
imp = pd.DataFrame(rows)
print(f"Median R-amp loss {imp['amp_loss_%'].median():.1f}% | median QRS-upslope loss {imp['slope_loss_%'].median():.1f}% "
      "(slope carries the delta wave; ~16% is a grey zone, motivated testing 0.5-75 -> ruled out by probes).")

### Block 4.8a: Probe M1 (NeuroKit) — enriched extraction [guarded]

In [ ]:
# PROBE M1 (NeuroKit2, lead II) — enriched extraction under 3 candidate filters (no_filter / 0.5-40 /
# 0.5-75). GUARDED: skips if probe CSVs exist (NeuroKit delineation is slow). Writes filter_probes/.
import warnings, numpy as np, pandas as pd, neurokit2 as nk
from scipy.signal import butter, sosfiltfilt
from tqdm import tqdm
warnings.filterwarnings('ignore')

FILTERS = {'no_filter': None, 'bp_0.5_40': (0.5, 40), 'bp_0.5_75': (0.5, 75)}
LEAD_II = LEADS_CANONICAL.index('II')
SOS = {k: (butter(4,[v[0]/(FS/2),v[1]/(FS/2)],btype='band',output='sos') if v else None) for k,v in FILTERS.items()}
FEAT_COLS = ['PR_ms','QRS_ms','QT_ms','P_ms','PR_seg_ms','ST_seg_ms','P_amp','Q_amp','R_amp','S_amp','T_amp',
             'QR_ratio','RS_ratio','QRS_rise_ms','QRS_upslope','QRS_delta_slope','RR_mean','RR_std','HR','n_beats']

def safe_med(a):
    a = np.array(a, float); a = a[~np.isnan(a)]; return np.nanmedian(a) if len(a) else np.nan

def extract_nk_extended(sig):
    sig = np.ascontiguousarray(sig, dtype=np.float64)
    _, rp = nk.ecg_peaks(sig, sampling_rate=FS); rp = rp['ECG_R_Peaks']
    if len(rp) < 3: return None
    _, w = nk.ecg_delineate(sig, rpeaks=rp, sampling_rate=FS, method='dwt')
    def arr(k): return np.array(w.get(k, []), float)
    P_on,P_off,P_pk = arr('ECG_P_Onsets'),arr('ECG_P_Offsets'),arr('ECG_P_Peaks')
    R_on,R_off = arr('ECG_R_Onsets'),arr('ECG_R_Offsets')
    Q_pk,S_pk,T_off,T_pk = arr('ECG_Q_Peaks'),arr('ECG_S_Peaks'),arr('ECG_T_Offsets'),arr('ECG_T_Peaks')
    rpa = np.array(rp, float)
    def dur_ms(a,b):
        n=min(len(a),len(b))
        if n==0: return np.nan
        d=(b[:n]-a[:n]); d=d[~np.isnan(d)]; return safe_med(d)/FS*1000 if len(d) else np.nan
    def amp_at(idx):
        idx=idx[~np.isnan(idx)].astype(int); idx=idx[(idx>=0)&(idx<len(sig))]; return safe_med(sig[idx]) if len(idx) else np.nan
    PR,QRS,QT = dur_ms(P_on,R_on),dur_ms(R_on,R_off),dur_ms(R_on,T_off)
    Pdur,PRseg,STseg = dur_ms(P_on,P_off),dur_ms(P_off,R_on),dur_ms(R_off,T_pk if len(T_pk) else T_off)
    Pa,Qa,Ra,Sa,Ta = amp_at(P_pk),amp_at(Q_pk),amp_at(rpa),amp_at(S_pk),amp_at(T_pk)
    rises,slopes,dslopes=[],[],[]; n=min(len(R_on),len(rpa))
    for i in range(n):
        on,pk=R_on[i],rpa[i]
        if np.isnan(on) or np.isnan(pk): continue
        on,pk=int(on),int(pk)
        if pk<=on or pk>=len(sig): continue
        rises.append((pk-on)/FS*1000); slopes.append((sig[pk]-sig[on])/((pk-on)/FS) if pk>on else np.nan)
        third=on+max(1,(pk-on)//3); dslopes.append((sig[third]-sig[on])/((third-on)/FS) if third>on else np.nan)
    rr=np.diff(rpa)/FS*1000
    return {'PR_ms':PR,'QRS_ms':QRS,'QT_ms':QT,'P_ms':Pdur,'PR_seg_ms':PRseg,'ST_seg_ms':STseg,
            'P_amp':Pa,'Q_amp':Qa,'R_amp':Ra,'S_amp':Sa,'T_amp':Ta,
            'QR_ratio':(Qa/Ra if Ra and not np.isnan(Ra) else np.nan),
            'RS_ratio':(Ra/Sa if Sa and not np.isnan(Sa) and Sa!=0 else np.nan),
            'QRS_rise_ms':safe_med(rises),'QRS_upslope':safe_med(slopes),'QRS_delta_slope':safe_med(dslopes),
            'RR_mean':np.nanmean(rr) if len(rr) else np.nan,'RR_std':np.nanstd(rr) if len(rr) else np.nan,
            'HR':60000/np.nanmean(rr) if len(rr) and np.nanmean(rr)>0 else np.nan,'n_beats':len(rpa)}

need = [PROBES / f"probe_m1_ext_{f}.csv" for f in FILTERS]
if all(p.exists() for p in need) and not FORCE_REEXTRACT:
    print("probe_m1_ext_*.csv exist -> SKIPPED M1 NeuroKit extraction (loaded from cache).")
else:
    tr = meta[meta.fold.between(1,8)]; parts=[tr[tr.label==1]]
    for s in tr.source.unique():
        neg=tr[(tr.source==s)&(tr.label==0)]; parts.append(neg.sample(n=min(4000,len(neg)), random_state=42))
    sample = pd.concat(parts).drop_duplicates('ecg_id').reset_index(drop=True)
    for fname, sos in SOS.items():
        rows=[]
        for _, r in tqdm(sample.iterrows(), total=len(sample), desc=f"M1 [{fname}]", unit="ecg"):
            try:
                sig = load_signal(r['ecg_id'], r['source'])[:, LEAD_II]
                if sos is not None: sig = sosfiltfilt(sos, np.ascontiguousarray(sig, dtype=np.float64))
                f = extract_nk_extended(sig) or {k: np.nan for k in FEAT_COLS}
            except Exception:
                f = {k: np.nan for k in FEAT_COLS}
            f.update({'ecg_id':r['ecg_id'],'source':r['source'],'label':r['label'],'fold':r['fold']})
            rows.append(f)
        pd.DataFrame(rows).to_csv(PROBES / f"probe_m1_ext_{fname}.csv", index=False)
    print("M1 extraction done -> probe_m1_ext_*.csv")

### Block 4.8b: Probe M1 — decisive same/cross/combined grid

In [ ]:
# PROBE M1 — decisive grid: AUC per filter x configuration (combined CV folds 1-8, same-dataset CV,
# and cross-dataset PTB<->Ningbo). 6 common form/delta features, XGBoost (NaN handled natively).
import numpy as np, pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
FILTERS = ['no_filter','bp_0.5_40','bp_0.5_75']
FEATS = ['QRS_upslope','S_amp','R_amp','T_amp','PR_ms','PR_seg_ms']

def make_xgb(spw):
    return XGBClassifier(n_estimators=100,max_depth=2,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,
        reg_lambda=2.0,min_child_weight=3,scale_pos_weight=spw,eval_metric='auc',tree_method='hist',random_state=42,n_jobs=-1)
def cv_same(data, feats):
    d=data[data.fold.between(1,8)].reset_index(drop=True); y=d.label.values; oof=np.zeros(len(d))
    spw=(y==0).sum()/max((y==1).sum(),1)
    for h in sorted(d.fold.unique()):
        tr,va=d.fold!=h,d.fold==h
        if d.loc[va,'label'].sum()==0 or d.loc[tr,'label'].sum()==0: continue
        oof[va.values]=make_xgb(spw).fit(d.loc[tr,feats],y[tr.values]).predict_proba(d.loc[va,feats])[:,1]
    return y, oof
def cross(tr_d, te_d, feats):
    tr=tr_d[tr_d.fold.between(1,8)]; te=te_d[te_d.fold.between(1,8)]
    spw=(tr.label==0).sum()/max((tr.label==1).sum(),1)
    m=make_xgb(spw).fit(tr[feats], tr.label); return te.label.values, m.predict_proba(te[feats])[:,1]

rows=[]
for fname in FILTERS:
    df=pd.read_csv(PROBES / f"probe_m1_ext_{fname}.csv", dtype={'ecg_id':str})
    ptb,nin=df[df.source=='ptbxl'],df[df.source=='ningbo']
    cfgs={'combined (CV)':cv_same(df,FEATS),'PTB->PTB (CV)':cv_same(ptb,FEATS),'Ning->Ning (CV)':cv_same(nin,FEATS),
          'PTB->Ning (cross)':cross(ptb,nin,FEATS),'Ning->PTB (cross)':cross(nin,ptb,FEATS)}
    for cfg,(y,p) in cfgs.items():
        rows.append({'filter':fname,'config':cfg,'AUC':roc_auc_score(y,p),'n_WPW':int(y.sum())})
res=pd.DataFrame(rows)
pd.set_option('display.float_format', lambda x:f'{x:.4f}')
for fname in FILTERS:
    print(f"\n--- {fname} ---"); print(res[res['filter']==fname][['config','AUC','n_WPW']].to_string(index=False))
print("\nCross vs same drop per filter:")
for fname in FILTERS:
    s=res[res['filter']==fname].set_index('config')['AUC']
    same=np.mean([s['PTB->PTB (CV)'],s['Ning->Ning (CV)']]); crs=np.mean([s['PTB->Ning (cross)'],s['Ning->PTB (cross)']])
    print(f"  {fname:12s}: same~{same:.3f} cross~{crs:.3f} drop={same-crs:+.3f} combined={s['combined (CV)']:.3f}")
print("Verdict M1: 0.5-40 best combined; small cross-same drop -> fusion generalizes; QRS_upslope is the delta signature.")

### Block 4.9: Probe M3 (wavelet) — extraction [guarded] + grid

In [ ]:
# PROBE M3 (wavelet db4, lead II) — extraction under 3 filters (GUARDED), FDR feature gate, then the
# same/cross/combined AUC grid. A radically different (frequency) family to test the filter/fusion decision.
import numpy as np, pandas as pd, pywt
from scipy.signal import butter, sosfiltfilt
from scipy.stats import entropy as scipy_entropy, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from collections import Counter
from tqdm import tqdm

FILTERS = {'no_filter': None, 'bp_0.5_40': (0.5, 40), 'bp_0.5_75': (0.5, 75)}
LEAD_II = LEADS_CANONICAL.index('II'); WAVELET = 'db4'; LEVELS = 4
LEVEL_NAMES = ['cA4','cD4','cD3','cD2','cD1']
FEAT_ALL = [f'{n}_{s}' for n in LEVEL_NAMES for s in ['energy','energy_rel','std','mean_abs','entropy']]
SOS = {k: (butter(4,[v[0]/(FS/2),v[1]/(FS/2)],btype='band',output='sos') if v else None) for k,v in FILTERS.items()}

def wavelet_feats(sig):
    sig = np.ascontiguousarray(sig, dtype=np.float64)
    coeffs = pywt.wavedec(sig, WAVELET, level=LEVELS); total = sum(np.sum(c**2) for c in coeffs) + 1e-12
    o = {}
    for name, c in zip(LEVEL_NAMES, coeffs):
        e = np.sum(c**2); p = c**2/(np.sum(c**2)+1e-12)
        o[f'{name}_energy']=e; o[f'{name}_energy_rel']=e/total; o[f'{name}_std']=np.std(c)
        o[f'{name}_mean_abs']=np.mean(np.abs(c)); o[f'{name}_entropy']=scipy_entropy(p+1e-12)
    return o

need = [PROBES / f"probe_m3_{f}.csv" for f in FILTERS]
if all(p.exists() for p in need) and not FORCE_REEXTRACT:
    print("probe_m3_*.csv exist -> SKIPPED M3 wavelet extraction (loaded from cache).")
else:
    tr = meta[meta.fold.between(1,8)]; parts=[tr[tr.label==1]]
    for s in tr.source.unique():
        neg=tr[(tr.source==s)&(tr.label==0)]; parts.append(neg.sample(n=min(4000,len(neg)), random_state=42))
    sample = pd.concat(parts).drop_duplicates('ecg_id').reset_index(drop=True)
    for fname, sos in SOS.items():
        rows=[]
        for _, r in tqdm(sample.iterrows(), total=len(sample), desc=f"M3 [{fname}]", unit="ecg"):
            try:
                sig = load_signal(r['ecg_id'], r['source'])[:, LEAD_II]
                if sos is not None: sig = sosfiltfilt(sos, np.ascontiguousarray(sig, dtype=np.float64))
                f = wavelet_feats(sig)
            except Exception:
                f = {k: np.nan for k in FEAT_ALL}
            f.update({'ecg_id':r['ecg_id'],'source':r['source'],'label':r['label'],'fold':r['fold']})
            rows.append(f)
        pd.DataFrame(rows).to_csv(PROBES / f"probe_m3_{fname}.csv", index=False)
    print("M3 extraction done -> probe_m3_*.csv")

def cohens_d(a,b):
    a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return np.nan
    sp=np.sqrt((a.std(ddof=1)**2+b.std(ddof=1)**2)/2); return (a.mean()-b.mean())/sp if sp>0 else np.nan
def make_xgb(spw): return XGBClassifier(n_estimators=100,max_depth=2,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,reg_lambda=2.0,min_child_weight=3,scale_pos_weight=spw,eval_metric='auc',tree_method='hist',random_state=42,n_jobs=-1)
def cv_same(d,feats):
    d=d[d.fold.between(1,8)].reset_index(drop=True); y=d.label.values; oof=np.zeros(len(d)); spw=(y==0).sum()/max((y==1).sum(),1)
    for h in sorted(d.fold.unique()):
        tr,va=d.fold!=h,d.fold==h
        if d.loc[va,'label'].sum()==0 or d.loc[tr,'label'].sum()==0: continue
        oof[va.values]=make_xgb(spw).fit(d.loc[tr,feats],y[tr.values]).predict_proba(d.loc[va,feats])[:,1]
    return y,oof
def cross(a,b,feats):
    tr=a[a.fold.between(1,8)]; te=b[b.fold.between(1,8)]; spw=(tr.label==0).sum()/max((tr.label==1).sum(),1)
    return te.label.values, make_xgb(spw).fit(tr[feats],tr.label).predict_proba(te[feats])[:,1]

# Feature gate (FDR) per filter, keep features passing in >=2/3 filters
sel={}
for fname in FILTERS:
    df=pd.read_csv(PROBES/f"probe_m3_{fname}.csv",dtype={'ecg_id':str}); trd=df[df.fold.between(1,8)].copy()
    for c in FEAT_ALL: trd[c]=trd[c].fillna(trd[c].median())
    w,n=trd[trd.label==1],trd[trd.label==0]; ps=[]; ds=[]
    for c in FEAT_ALL:
        ds.append(cohens_d(w[c].values,n[c].values))
        try: ps.append(mannwhitneyu(w[c].dropna(),n[c].dropna())[1])
        except: ps.append(np.nan)
    r=pd.DataFrame({'f':FEAT_ALL,'d':ds,'p':ps}); ok=r.p.notna(); r.loc[ok,'pf']=multipletests(r.loc[ok,'p'],method='fdr_bh')[1]
    sel[fname]=set(r[(r.d.abs()>0.3)&(r.pf<0.05)]['f'])
cnt=Counter(f for s in sel.values() for f in s); FEATS=sorted([f for f,c in cnt.items() if c>=2])
print(f"M3 common features (>=2/3 filters): {len(FEATS)}")

rows=[]
for fname in FILTERS:
    df=pd.read_csv(PROBES/f"probe_m3_{fname}.csv",dtype={'ecg_id':str}); ptb,nin=df[df.source=='ptbxl'],df[df.source=='ningbo']
    for cfg,(y,p) in {'combined (CV)':cv_same(df,FEATS),'PTB->PTB (CV)':cv_same(ptb,FEATS),'Ning->Ning (CV)':cv_same(nin,FEATS),
                      'PTB->Ning (cross)':cross(ptb,nin,FEATS),'Ning->PTB (cross)':cross(nin,ptb,FEATS)}.items():
        rows.append({'filter':fname,'config':cfg,'AUC':roc_auc_score(y,p)})
res=pd.DataFrame(rows); pd.set_option('display.float_format', lambda x:f'{x:.4f}')
for fname in FILTERS:
    s=res[res['filter']==fname].set_index('config')['AUC']
    same=np.mean([s['PTB->PTB (CV)'],s['Ning->Ning (CV)']]); crs=np.mean([s['PTB->Ning (cross)'],s['Ning->PTB (cross)']])
    print(f"  {fname:12s}: combined {s['combined (CV)']:.3f} | same~{same:.3f} cross~{crs:.3f} drop={same-crs:+.3f}")
print("Verdict M3: generalizes (drop ~0); FDR gate spontaneously drops cD1/cD2 (the batch-effect bands).")

### Block 4.10: Probe M7 (1D-CNN) — training [guarded] + grid

In [ ]:
# PROBE M7 (1D-CNN end-to-end, lead II z-scored). GUARDED: the 48-min training is skipped if
# probe_m7_grid.csv exists (its cached results are normalized to English labels on load). A deep
# end-to-end family, third independent check of the filter/fusion decision.
import os, json, random
import numpy as np, pandas as pd, torch
import torch.nn as nn
from scipy.signal import butter, sosfiltfilt
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

torch.set_num_threads(10); DEVICE = torch.device('cpu')
FILTERS = {'no_filter': None, 'bp_0.5_40': (0.5, 40), 'bp_0.5_75': (0.5, 75)}
SEEDS = [0, 1, 2]; LEAD_II = LEADS_CANONICAL.index('II'); GRID_CSV = PROBES / "probe_m7_grid.csv"

class SmallECGCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1,8,15,stride=2,padding=7), nn.BatchNorm1d(8), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(8,16,11,stride=1,padding=5), nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(16,32,7,stride=1,padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.AdaptiveMaxPool1d(8))
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.5), nn.Linear(32*8,32), nn.ReLU(),
                                  nn.Dropout(0.5), nn.Linear(32,1))
    def forward(self, x): return self.head(self.features(x)).squeeze(1)

NORM = {'sans_filtre':'no_filter', 'combiné':'combined', 'PTB→PTB':'PTB->PTB',
        'Ning→Ning':'Ning->Ning', 'PTB→Ning':'PTB->Ning', 'Ning→PTB':'Ning->PTB'}

if GRID_CSV.exists() and not FORCE_REEXTRACT:
    res = pd.read_csv(GRID_CSV)
    res = res.rename(columns={'filtre': 'filter'})
    res['filter'] = res['filter'].replace(NORM); res['config'] = res['config'].replace(NORM)
    res.to_csv(GRID_CSV, index=False)   # one-time normalization of the cached grid to English labels
    print("probe_m7_grid.csv exists -> SKIPPED CNN training (~48 min); cached results normalized to English.")
else:
    def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s)
    SOS = {k: (butter(4,[v[0]/(FS/2),v[1]/(FS/2)],btype='band',output='sos') if v else None) for k,v in FILTERS.items()}
    tr = meta[meta.fold.between(1,8)]; parts=[tr[tr.label==1]]
    for s in tr.source.unique():
        neg=tr[(tr.source==s)&(tr.label==0)]; parts.append(neg.sample(n=min(4000,len(neg)), random_state=42))
    sample = pd.concat(parts).drop_duplicates('ecg_id').reset_index(drop=True)
    N=len(sample); raw=np.zeros((N,5000),dtype=np.float32); ok=np.ones(N,bool)
    for i,(_,r) in enumerate(tqdm(sample.iterrows(), total=N, desc="Load raw", unit="ecg")):
        try: raw[i]=load_signal(r['ecg_id'], r['source'])[:, LEAD_II]
        except Exception: ok[i]=False
    y=sample.label.values.astype(np.float32); fold=sample.fold.values; source=sample.source.values
    SIGNALS={}
    for fname,sos in SOS.items():
        X=np.zeros((N,5000),dtype=np.float32)
        for i in range(N):
            if not ok[i]: continue
            s=raw[i].astype(np.float64); X[i]=(sosfiltfilt(sos,np.ascontiguousarray(s)) if sos is not None else s).astype(np.float32)
        mu=X.mean(1,keepdims=True); sd=X.std(1,keepdims=True)+1e-6; SIGNALS[fname]=(X-mu)/sd
    def train_eval(Xtr,ytr,Xte,yte,seed,max_epochs=40,patience=6,bs=128):
        set_seed(seed); idx=np.arange(len(ytr)); rng=np.random.RandomState(seed); rng.shuffle(idx)
        pos,neg=idx[ytr[idx]==1],idx[ytr[idx]==0]; nvp,nvn=max(1,int(0.1*len(pos))),int(0.1*len(neg))
        vidx=np.concatenate([pos[:nvp],neg[:nvn]]); tidx=np.concatenate([pos[nvp:],neg[nvn:]])
        Xt=torch.tensor(Xtr[tidx]).unsqueeze(1); yt=torch.tensor(ytr[tidx])
        Xv=torch.tensor(Xtr[vidx]).unsqueeze(1); yv=torch.tensor(ytr[vidx]); Xe=torch.tensor(Xte).unsqueeze(1)
        net=SmallECGCNN().to(DEVICE); pw=torch.tensor([(ytr==0).sum()/max((ytr==1).sum(),1)],dtype=torch.float32)
        loss_fn=nn.BCEWithLogitsLoss(pos_weight=pw); opt=torch.optim.Adam(net.parameters(),lr=1e-3,weight_decay=1e-4)
        best,bs_state,wait=-1,None,0; n=len(tidx)
        for ep in range(max_epochs):
            net.train(); perm=torch.randperm(n)
            for i in range(0,n,bs):
                b=perm[i:i+bs]; opt.zero_grad(); loss_fn(net(Xt[b]),yt[b]).backward(); opt.step()
            net.eval()
            with torch.no_grad(): av=roc_auc_score(yv.numpy(),torch.sigmoid(net(Xv)).numpy()) if yv.sum()>0 else 0.5
            if av>best: best,bs_state,wait=av,{k:v.clone() for k,v in net.state_dict().items()},0
            else: wait+=1
            if wait>=patience: break
        net.load_state_dict(bs_state); net.eval()
        with torch.no_grad(): pe=torch.sigmoid(net(Xe)).numpy()
        return roc_auc_score(yte,pe) if yte.sum()>0 else np.nan
    def masks():
        t18=np.isin(fold,range(1,9))
        return {'combined':((t18,t18,'cv')),'PTB->PTB':(((source=='ptbxl')&t18,(source=='ptbxl')&t18,'cv')),
                'Ning->Ning':(((source=='ningbo')&t18,(source=='ningbo')&t18,'cv')),
                'PTB->Ning':(((source=='ptbxl')&t18,(source=='ningbo')&t18,'cross')),
                'Ning->PTB':(((source=='ningbo')&t18,(source=='ptbxl')&t18,'cross'))}
    def split_cv(mask,seed):
        idx=np.where(mask)[0]; rng=np.random.RandomState(seed); pos,neg=idx[y[idx]==1],idx[y[idx]==0]
        rng.shuffle(pos); rng.shuffle(neg); ntp,ntn=int(0.8*len(pos)),int(0.8*len(neg))
        return np.concatenate([pos[:ntp],neg[:ntn]]), np.concatenate([pos[ntp:],neg[ntn:]])
    rows=[]
    for fname in FILTERS:
        X=SIGNALS[fname]
        for cfg,(mtr,mte,kind) in masks().items():
            aucs=[]
            for seed in SEEDS:
                if kind=='cv': tr_i,te_i=split_cv(mtr,seed); a=train_eval(X[tr_i],y[tr_i],X[te_i],y[te_i],seed)
                else: a=train_eval(X[mtr],y[mtr],X[mte],y[mte],seed)
                aucs.append(a)
            rows.append({'filter':fname,'config':cfg,'AUC_mean':np.nanmean(aucs),'AUC_std':np.nanstd(aucs)})
    res=pd.DataFrame(rows); res.to_csv(GRID_CSV,index=False)
    print("M7 CNN training done -> probe_m7_grid.csv")

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
for fname in ['no_filter','bp_0.5_40','bp_0.5_75']:
    s=res[res['filter']==fname].set_index('config')['AUC_mean']
    same=np.mean([s['PTB->PTB'],s['Ning->Ning']]); crs=np.mean([s['PTB->Ning'],s['Ning->PTB']])
    print(f"  {fname:12s}: combined {s['combined']:.3f} | same~{same:.3f} cross~{crs:.3f} drop={same-crs:+.3f}")
print("Verdict M7: powerful but unstable (high seed variance); still generalizes -> learns WPW, not the dataset.")

### Block 4.11: Synthesis — freeze the filter & fusion decision

In [ ]:
# SYNTHESIS — freeze the filter & fusion decision. The three probes (clinical M1, frequency M3, deep M7)
# agree: filtering helps, 0.5-40 Hz is the only positively-designated cutoff, and fusion generalizes
# cross-dataset despite a ~0.95 batch effect. Writes the frozen filter_config.json (status DECIDED).
import json
config = {
    "status": "DECIDED — 0.5-40 Hz bandpass retained, PTB-XL+Ningbo fusion validated (3 concordant probes)",
    "filter_FINAL": {"type": "butterworth", "order": 4, "low": 0.5, "high": 40, "fs": 500,
                     "phase": "zero-phase (sosfiltfilt)",
                     "implementation": "scipy.signal.butter(4,[0.5,40]/(fs/2),'band','sos') + sosfiltfilt on float64"},
    "fusion": "VALIDATED — combined PTB-XL+Ningbo models (no per-dataset fallback)",
    "decision_basis": {
        "batch_effect": "dataset-separation AUC ~0.95 (strong), NOT harmonizable by any bandpass (~0.05 drop). Filtering is denoising, not harmonization.",
        "true_judge": "cross-dataset generalization (train one dataset -> test the other), NOT the raw batch effect",
        "probes": {
            "M1_neurokit_clinical": {"combined_0.5_40": 0.783, "cross_same_drop": "+0.02",
                "note": "designates 0.5-40; QRS_upslope d=-0.61 (delta signature) maximized at 0.5-40"},
            "M3_wavelet_freq": {"combined_0.5_40": 0.697, "cross_same_drop": "-0.004",
                "note": "generalizes; gate spontaneously drops cD1/cD2 (batch-effect bands)"},
            "M7_cnn_deep": {"combined_0.5_40": 0.867, "cross_same_drop": "+0.07",
                "note": "stronger but unstable (+/-0.13 across seeds); still generalizes"}},
        "verdict_filter": "Unanimous that filtering helps. M1 designates 0.5-40; M3 and M7 judge it equivalent to 0.5-75 (no probe reliably prefers 0.5-75). 0.5-40 retained: only positively designated + ECG standard.",
        "verdict_fusion": "All 3 families (clinical/frequency/deep) generalize cross-dataset despite batch effect 0.95 -> they learn WPW, not the dataset. Fusion is robust."},
    "implications": {
        "common_filter": "0.5-40 applied identically to M1-M5, M7 and Flask (train/inference consistency)",
        "cnn_normalization": "per-signal z-score for M7 (removes the inter-dataset amplitude gap at the source)",
        "nan_neurokit": "extraction failure 7x more frequent in WPW (delta blurs R_Onset) -> 'extraction_failed' feature for M1; XGBoost handles NaN natively"},
    "probe_files": "data/processed/filter_probes/probe_m1_ext_*.csv, probe_m3_*.csv, probe_m7_grid.csv (exploratory)"}

out = PROCESSED / "filter_config.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print(f"DECISION FROZEN -> {out}")
print("  Filter: Butterworth order 4, 0.5-40 Hz, zero-phase | Fusion: VALIDATED (3 concordant probes)")